# Commercial investigation vertical extraction

Load the **commercial_investigation** subset from intent-classified data, then extract **business verticals** (e.g. sports, health, real estate) using:
1. **IAB Content Taxonomy** keyword matching (all queries + all answers)
2. **LLM-based** structured output with intent re-classification (parallel, cached)
3. **Embedding-based** similarity to reference verticals
4. **Rule-based** vertical keywords (optional)

Uses **all user messages (all queries)** and **all assistant messages (all answers)** per conversation.

In [1]:
# Control variables (edit and run first)
INTENT_OUTPUT_DIR = "intent_output"
USE_SAMPLE = True
SAMPLE_N = 2000
RANDOM_SEED = 42
RUN_TAXONOMY = True
RUN_LLM_VERTICAL = True
RUN_EMBEDDING_VERTICAL = True
RUN_RULE_VERTICAL = True
IAB_DATA_DIR = None   # e.g. Path("data") to use data/iab_content_taxonomy_tier1_tier2.csv
VERTICAL_LLM_CACHE_PATH = "intent_output/vertical_intent_llm.parquet"
LLM_BATCH_SIZE = 150
LLM_MAX_WORKERS = 150  # set batch_size >= max_workers to use all workers; rate limits reported after run

## Load commercial_investigation and add text columns

Load the commercial_investigation parquet, ensure conversation is parsed, optionally sample, then add **all_queries** and **all_answers** (all user and all assistant messages per conversation).

In [2]:
from pathlib import Path
import pandas as pd

from eda_utils import ensure_conversation_parsed
from intent_analysis_utils import ensure_conversation_normalized
from intent_taxonomy import load_category
from commercial_vertical_utils import add_all_queries_answers_columns

df = load_category(INTENT_OUTPUT_DIR, "commercial_investigation", which="major")
df = ensure_conversation_parsed(df)
df = ensure_conversation_normalized(df)
if USE_SAMPLE and len(df) > SAMPLE_N:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
df = add_all_queries_answers_columns(df, conversation_col="conversation")
print(f"Loaded {len(df)} commercial_investigation rows. Columns: {list(df.columns)}")
df[["conversation_id", "all_queries", "all_answers"]].head(2)

Loaded 2000 commercial_investigation rows. Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text', 'intent_major', 'intent_sub', 'all_queries', 'all_answers']


,conversation_id,all_queries,all_answers
0,dd7456f94191d7684daf0309055c277d,create a humor immaculately detailed Scott the...,"I'm sorry, I cannot generate inappropriate or ..."
1,3e7bdb47db4b98df9d4d2fae1191eed0,If a female friend does wear a dress size 8. I...,It is difficult to accurately estimate someone...


## Taxonomy-based vertical (IAB keyword match)

Match keywords from **all_queries + all_answers** to IAB Content Taxonomy (Tier 1 / Tier 2). Uses local CSV in `data/` if `IAB_DATA_DIR` is set, otherwise fetches from IAB GitHub or uses embedded fallback.

In [3]:
if RUN_TAXONOMY:
    from commercial_vertical_utils import assign_vertical_iab

    data_dir = Path(IAB_DATA_DIR) if IAB_DATA_DIR else None
    df = assign_vertical_iab(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
        data_dir=data_dir,
    )
    print("Vertical (IAB) Tier 1 value counts:")
    display(df["vertical_tier1_iab"].value_counts().head(15))

Vertical (IAB) Tier 1 value counts:


vertical_tier1_iab
Family and Relationships               684
Personal Celebrations & Life Events    277
Style & Fashion                        161
Sensitive Topics                       149
Home & Garden                          104
Hobbies & Interests                    103
Real Estate                             98
Food & Drink                            93
Sports                                  61
Healthy Living                          48
Education                               31
Medical Health                          25
Books and Literature                    24
Genres                                  21
Personal Finance                        21
Name: count, dtype: int64

## LLM-based vertical and intent re-classification

Call OpenAI with **all_queries + all_answers** per conversation; get structured JSON with `vertical_tier1`, `vertical_tier2`, and `intent_revised`. Runs in parallel with caching.

In [4]:
if RUN_LLM_VERTICAL:
    from dotenv import load_dotenv
    load_dotenv()

    from commercial_vertical_utils import label_vertical_intent_llm_parallel

    df = label_vertical_intent_llm_parallel(
        df,
        queries_col="all_queries",
        answers_col="all_answers",
        cache_path=VERTICAL_LLM_CACHE_PATH,
        use_cache=True,
        batch_size=LLM_BATCH_SIZE,
        max_workers=LLM_MAX_WORKERS,
    )
    print("Vertical (LLM) Tier 1 value counts:")
    display(df["vertical_tier1_llm"].value_counts().head(15))
    print("Intent revised vs intent_major:")
    display(pd.crosstab(df["intent_major"], df["intent_revised"], margins=True))

Vertical (LLM) Tier 1 value counts:


vertical_tier1_llm
Other            792
Technology       327
Education        323
Health           275
Sports            97
Travel            40
Shopping          34
Finance           34
Food & Drink      34
Automotive        11
Real Estate        8
Music              8
Art                3
Entertainment      3
Gaming             2
Name: count, dtype: int64

Intent revised vs intent_major:


intent_revised,commercial_investigation,informational,navigational,transactional,All
intent_major,,,,,
commercial_investigation,42,1939,10,9,2000
All,42,1939,10,9,2000


## Spot check: conversations by LLM vertical

Sample a few conversations for a **vertical of your choice** (LLM-assigned `vertical_tier1_llm`). Adjust `SPOTCHECK_VERTICAL` and `N_PER_VERTICAL` below, then run the cell.

In [5]:
# Spot check by vertical: pick a vertical and how many conversations to show
SPOTCHECK_VERTICAL = "Technology"   # change to any vertical_tier1_llm value (e.g. "Health", "Sports", "Other")
N_PER_VERTICAL = 3                 # number of conversations to print (adjustable)

if "vertical_tier1_llm" in df.columns:
    from intent_analysis_utils import sample_by_category, format_conversation
    available = sorted(df["vertical_tier1_llm"].dropna().unique().tolist())
    print(f"Available verticals: {available}\n")
    subset = df[df["vertical_tier1_llm"] == SPOTCHECK_VERTICAL]
    if len(subset) == 0:
        print(f"No rows with vertical_tier1_llm == {SPOTCHECK_VERTICAL!r}. Choose from: {available}")
    else:
        n = min(N_PER_VERTICAL, len(subset))
        sampled = subset.sample(n=n, random_state=RANDOM_SEED)
        print(f"Vertical: {SPOTCHECK_VERTICAL} (showing {n} of {len(subset)} conversations)\n")
        print("=" * 60)
        for idx, row in sampled.iterrows():
            print(f"--- conversation_id: {row['conversation_id']} ---")
            print(format_conversation(row["conversation"], width=80))
            print("=" * 60)
else:
    print("Run the LLM-based vertical cell first so vertical_tier1_llm exists.")

Available verticals: ['Art', 'Automotive', 'Beauty', 'Business', 'Economics', 'Education', 'Entertainment', 'Fashion', 'Finance', 'Food & Drink', 'Gaming', 'Health', 'Home', 'Home Improvement', 'Legal', 'Music', 'Other', 'Pets', 'Politics', 'Real Estate', 'Shopping', 'Sports', 'Technology', 'Travel']

Vertical: Technology (showing 3 of 327 conversations)

--- conversation_id: d30e69ba51d3322ae58936b5b8d8895a ---
[user]
Hi there! Send best ideas for B2B startup

[assistant]
1. AI-powered lead generation platform: Develop an artificial intelligence (AI)
powered platform that identifies, scores, and matches prospective customers and
businesses with relevant products, services, or partners.  2. Virtual events and
webinars platform: Create a virtual event platform specifically designed for B2B
businesses enabling professionals to attend online workshops, webinars, and
conferences, as well as network and collaborate on various projects.  3. On-
demand B2B consulting services: Launch an onlin

## Spot check: re-classified intent by category

Sample a few conversations **per intent_revised** category to review how the LLM re-classified intent. Adjust `N_PER_INTENT` below, then run the cell.

In [6]:
# Spot check by intent_revised: how many conversations per intent category
N_PER_INTENT = 3   # number of conversations to print per intent_revised (adjustable)

if "intent_revised" in df.columns:
    from intent_analysis_utils import sample_by_category, format_conversation
    sampled = sample_by_category(
        df, category_col="intent_revised", n_per_cat=N_PER_INTENT, seed=RANDOM_SEED
    )
    for intent_cat in sampled["intent_revised"].dropna().unique():
        block = sampled[sampled["intent_revised"] == intent_cat]
        n_show = len(block)
        total = len(df[df["intent_revised"] == intent_cat])
        print(f"\n{'=' * 60}")
        print(f"intent_revised: {intent_cat} (showing {n_show} of {total})")
        print("=" * 60)
        for _, row in block.iterrows():
            print(f"--- conversation_id: {row['conversation_id']} | intent_major: {row['intent_major']} ---")
            print(format_conversation(row["conversation"], width=80))
            print("-" * 60)
else:
    print("Run the LLM-based vertical cell first so intent_revised exists.")


intent_revised: commercial_investigation (showing 3 of 42)
--- conversation_id: 517212b531203520b72f16a9f33dbfb2 | intent_major: commercial_investigation ---
[user]
{'Count': 5, 'title': 'バイクライフ-バイクと人生をより豊かに(AI Photobook)', 'englishtitle': 'Bike
Life - Make Your Life More Fulfilling with Bikes(AI Photobook)',
'katakanaoftitle': 'バイクライフ', '7ofdescriptionkeyword': ['バイク', '人生', '自由', '天空',
'旅行', '仲間', '冒険'], 'english7ofdescriptionkeyword': ['bike', 'life', 'freedom',
'sky', 'travel', 'comrades', 'adventure'], 'kdpofintroduction': 'もっと気楽に自由にバイクを楽し
んでみたいと思いませんか？バイクライフを送ることは、あなたの人生にも大きな影響を与えることでしょう。本書では、バイクに乗って自由な空間を訪れたり、仲間とともに旅
行に出かけたり、新しい冒険を始めたりする素晴らしいバイクライフスタイルをご紹介しています。', 'englishkdpofintroduction':
'Want to have more fun and freedom with your bike? Living the bike life can have
a profound impact on your life. This book features an array of thrilling bike
life experiences, from exploring open spaces on your bike, to taking trips with
friends, to embarking on new adventures.', 'prompt':

## Embedding-based vertical

Embed **all_queries + all_answers** and assign the reference vertical with highest cosine similarity.

In [7]:
if RUN_EMBEDDING_VERTICAL:
    from commercial_vertical_utils import assign_vertical_embedding

    df = assign_vertical_embedding(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
    )
    print("Vertical (embedding) value counts:")
    display(df["vertical_embedding"].value_counts().head(15))

/Users/Larry.Jin/miniconda3/envs/wildchat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2530.23it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2292.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-------------------

Vertical (embedding) value counts:


vertical_embedding
Health          375
Other           289
Technology      263
Sports          173
Shopping        166
Travel          163
Education       162
Food & Drink    137
Finance         115
Automotive       92
Real Estate      65
Name: count, dtype: int64

## Rule-based vertical (optional)

Assign vertical from curated keyword rules (first match wins) over **all_queries + all_answers**.

In [8]:
if RUN_RULE_VERTICAL:
    from commercial_vertical_utils import assign_vertical_rule_based

    df = assign_vertical_rule_based(
        df,
        text_col_queries="all_queries",
        text_col_answers="all_answers",
    )
    print("Vertical (rule) value counts:")
    display(df["vertical_rule"].value_counts().head(15))

Vertical (rule) value counts:


vertical_rule
Sports          846
Real Estate     420
Technology      301
Health          213
Other            64
Shopping         52
Automotive       50
Education        23
Travel           17
Finance          12
Food & Drink      2
Name: count, dtype: int64

## Summary: counts and agreement

Compare vertical distributions across methods and intent_revised vs intent_major.

In [9]:
summary_cols = ["vertical_tier1_iab", "vertical_tier1_llm", "vertical_embedding", "vertical_rule"]
existing = [c for c in summary_cols if c in df.columns]
if existing:
    for col in existing:
        print(f"--- {col} ---")
        display(df[col].value_counts().head(10))
if "intent_revised" in df.columns:
    print("Intent revised vs intent_major (crosstab):")
    display(pd.crosstab(df["intent_major"], df["intent_revised"], normalize="index").round(2))

--- vertical_tier1_iab ---


vertical_tier1_iab
Family and Relationships               684
Personal Celebrations & Life Events    277
Style & Fashion                        161
Sensitive Topics                       149
Home & Garden                          104
Hobbies & Interests                    103
Real Estate                             98
Food & Drink                            93
Sports                                  61
Healthy Living                          48
Name: count, dtype: int64

--- vertical_tier1_llm ---


vertical_tier1_llm
Other           792
Technology      327
Education       323
Health          275
Sports           97
Travel           40
Shopping         34
Finance          34
Food & Drink     34
Automotive       11
Name: count, dtype: int64

--- vertical_embedding ---


vertical_embedding
Health          375
Other           289
Technology      263
Sports          173
Shopping        166
Travel          163
Education       162
Food & Drink    137
Finance         115
Automotive       92
Name: count, dtype: int64

--- vertical_rule ---


vertical_rule
Sports         846
Real Estate    420
Technology     301
Health         213
Other           64
Shopping        52
Automotive      50
Education       23
Travel          17
Finance         12
Name: count, dtype: int64

Intent revised vs intent_major (crosstab):


intent_revised,commercial_investigation,informational,navigational,transactional
intent_major,,,,
commercial_investigation,0.02,0.97,0.0,0.0
